# 📈 Clase 2 — Visualización de distribuciones en Seaborn
## De `distplot` a `histplot` y `kdeplot`

**Situación:** El equipo pedagógico de una plataforma de cursos online quiere entender cuánto tiempo dedican los usuarios por sesión. Necesitan saber si la distribución es simétrica o sesgada, y si hay extremos que distorsionan la media.

**Objetivos:**
- Entender la evolución de `distplot` → `histplot` + `kdeplot`
- Crear histogramas con `histplot()` y controlar bins, color, stat
- Crear curvas de densidad con `kdeplot()` y controlar `bw_adjust`
- Combinar ambos con `histplot(kde=True)`
- Interpretar forma, sesgo, multimodalidad y outliers
- Comparar ambas técnicas y saber cuándo usar cada una

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Buena práctica: configurar estilo al inicio
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')
print(f'Seaborn {sns.__version__}')

---
## PARTE 1 — Introducción al análisis de distribuciones

### 1.1 ¿Qué nos dice la distribución de una variable?

In [ ]:
# Demostración: cuatro distribuciones con la misma media
np.random.seed(42)
n = 200

normal    = np.random.normal(50, 10, n)
sesgada   = np.random.exponential(15, n) + 20
bimodal   = np.concatenate([np.random.normal(35, 5, n//2),
                             np.random.normal(65, 5, n//2)])
uniforme  = np.random.uniform(20, 80, n)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('Cuatro distribuciones — misma media, formas distintas', fontweight='bold')

for ax, datos, titulo, color in zip(
    axes,
    [normal, sesgada, bimodal, uniforme],
    ['Normal (simétrica)','Sesgada (cola derecha)','Bimodal (2 picos)','Uniforme'],
    ['#2E75B6','#ED7D31','#70AD47','#7030A0']
):
    sns.histplot(datos, bins=20, kde=True, ax=ax, color=color, alpha=0.7)
    ax.axvline(np.mean(datos), color='red', linestyle='--', linewidth=2,
               label=f'Media {np.mean(datos):.0f}')
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print('Conclusión: la media no es suficiente. La FORMA de la distribución también importa.')

---
## PARTE 2 — De `distplot` a sus reemplazos modernos

### 2.1 `distplot` — función deprecada (contexto histórico)

In [ ]:
# Datos exactos de la presentación
data = pd.DataFrame({'edad': [23,25,29,31,34,36,38,40,42,45,47,49,52]})

# ⚠️ distplot está DEPRECADO desde Seaborn 0.11.0
# El siguiente código muestra el uso histórico — NO usar en proyectos nuevos
print('⚠️  distplot está DEPRECADO desde Seaborn 0.11.0')
print('   Se recomienda usar histplot() y kdeplot() de forma separada.')
print()
print('Código de la presentación (referencia histórica):')
print('''
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

data = pd.DataFrame({'edad': [23,25,29,31,34,36,38,40,42,45,47,49,52]})
sns.distplot(data['edad'])     # ← DEPRECADO
plt.title("Distribución de Edad")
plt.xlabel("Edad")
plt.ylabel("Densidad")
plt.show()
''')

### 2.2 `histplot()` — el reemplazo moderno del histograma

In [ ]:
# Código exacto de la presentación
sns.histplot(data=data, x='edad', bins=5, kde=False, color='skyblue')
plt.title('Histograma de Edad')
plt.show()

In [ ]:
# Visualizar proporciones relativas — exacto de la presentación
sns.histplot(data=data, x='edad', stat='probability', bins=5)
plt.title('Histograma de Edad — stat="probability"')
plt.ylabel('Probabilidad')
plt.show()

In [ ]:
# Explorar parámetro stat — todos los valores posibles
stats_disponibles = ['count', 'frequency', 'probability', 'proportion', 'density']

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
fig.suptitle('Parámetro stat= en histplot()', fontweight='bold')

for ax, stat in zip(axes, stats_disponibles):
    sns.histplot(data=data, x='edad', bins=5, stat=stat, ax=ax, color='#2E75B6')
    ax.set_title(f'stat="{stat}"', fontsize=9)
    ax.set_xlabel('Edad')

plt.tight_layout()
plt.show()

### 2.3 `kdeplot()` — curva de densidad

In [ ]:
# Código exacto de la presentación
sns.kdeplot(data=data['edad'], fill=True, color='orange')
plt.title('Curva de Densidad')
plt.show()

In [ ]:
# Efecto del parámetro bw_adjust — exacto de la presentación
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Efecto de bw_adjust en kdeplot() — suavizado de la curva', fontweight='bold')

for ax, bw in zip(axes, [0.3, 0.8, 1.5, 3.0]):
    sns.kdeplot(data=data['edad'], fill=True, bw_adjust=bw,
                color='orange', ax=ax)
    ax.set_title(f'bw_adjust={bw}', fontsize=10)
    ax.set_xlabel('Edad')
    ax.set_ylabel('Densidad')

plt.tight_layout()
plt.show()
print('bw_adjust bajo  → curva muy detallada (puede sobreajustarse al ruido)')
print('bw_adjust alto  → curva muy suavizada (puede ocultar estructura)')
print('Recomendado: 0.5–1.5 para datos de tamaño mediano')

### 2.4 Gráfico combinado — histplot + kde

In [ ]:
# Código exacto de la presentación
sns.histplot(data=data, x='edad', bins=5, kde=True, color='purple')
plt.title('Distribución combinada de Edad')
plt.show()

### 2.5 Tabla comparativa: histplot vs kdeplot

In [ ]:
# Tabla de la presentación
tabla_comp = pd.DataFrame({
    'Característica':     ['Representa','Forma visual','Tipo de datos',
                           'Interpretación','Requiere binning','Parámetros clave'],
    'Histograma (histplot)': ['Frecuencia o proporción','Barras','Discretos o continuos',
                              'Conteo en rangos (bins)','Sí',
                              'bins, stat, color, kde'],
    'Densidad (kdeplot)':    ['Estimación continua de densidad','Línea suave','Continuos',
                              'Probabilidad estimada','No',
                              'fill, bw_adjust, color']
})
print(tabla_comp.to_string(index=False))

### ✏️ Ejercicio 2 — Reflexiona:

In [ ]:
# ✏️ ¿Por qué se deprecó distplot? ¿Qué limitaciones tenía?
r_distplot = ""

# ✏️ ¿Cuándo preferirías kdeplot sobre histplot?
r_kde_vs_hist = ""

# ✏️ ¿Qué pasa si aplicas kdeplot a una variable categórica?
r_kde_cat = ""

# ✏️ ¿Cuándo podría ser problemático un bw_adjust muy bajo?
r_bw_bajo = ""

print(f'distplot deprecado: {r_distplot}')
print(f'kdeplot vs histplot: {r_kde_vs_hist}')
print(f'kdeplot en categórica: {r_kde_cat}')
print(f'bw_adjust bajo: {r_bw_bajo}')

---
## PARTE 3 — Actividad guiada: Minutos de conexión diaria

### 3.1 Cargar y explorar el dataset

In [ ]:
# Código exacto de la presentación
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='whitegrid')

data = pd.read_csv('conexiones_diarias.csv')
print(data.head(10))

In [ ]:
print('=== Estadísticas descriptivas ===')
print(data['min_conexion_dia'].describe().round(2))
print()
print(f'Media:    {data["min_conexion_dia"].mean():.2f} min')
print(f'Mediana:  {data["min_conexion_dia"].median():.2f} min')
print(f'Diferencia media-mediana: {data["min_conexion_dia"].mean() - data["min_conexion_dia"].median():.2f}')

### 3.2 Histograma simple (código exacto de la presentación)

In [ ]:
# Código exacto slide 20
sns.histplot(data=data, x='min_conexion_dia', bins=10, color='steelblue')
plt.title('Histograma de minutos de conexión diaria')
plt.xlabel('Minutos de conexión')
plt.ylabel('Frecuencia')
plt.show()

### 3.3 Curva de densidad (código exacto de la presentación)

In [ ]:
# Código exacto slide 21
sns.kdeplot(data=data['min_conexion_dia'], fill=True, bw_adjust=0.8, color='orange')
plt.title('Curva de densidad de minutos de conexión')
plt.xlabel('Minutos de conexión')
plt.ylabel('Densidad')
plt.show()

### 3.4 Visualización combinada (código exacto de la presentación)

In [ ]:
# Código exacto slide 22
sns.histplot(data=data, x='min_conexion_dia', bins=10, kde=True, color='purple', alpha=0.6)
plt.title('Distribución combinada de conexión diaria')
plt.xlabel('Minutos de conexión')
plt.ylabel('Frecuencia / Densidad')
plt.show()

In [ ]:
# Código completo del slide 24 — insumos finales
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('conexiones_diarias.csv')

# Histograma simple
sns.histplot(data=df, x='min_conexion_dia', bins=10)
plt.title('Histograma: Minutos de Conexión Diaria')
plt.show()

# Curva de densidad
sns.kdeplot(data=df['min_conexion_dia'], fill=True, bw_adjust=0.8, color='orange')
plt.title('Curva de Densidad: Minutos de Conexión Diaria')
plt.show()

# Gráfico combinado
sns.histplot(data=df, x='min_conexion_dia', kde=True, bins=10, color='skyblue')
plt.title('Distribución Combinada: Histograma + KDE')
plt.show()

### 3.5 Interpretación (preguntas del slide 23)

In [ ]:
# Diagnóstico cuantitativo de la forma
from scipy.stats import skew, kurtosis

sk = skew(df['min_conexion_dia'])
ku = kurtosis(df['min_conexion_dia'])

print('=== Diagnóstico de la distribución ===')
print(f'Sesgo (skewness):   {sk:.3f}  → {"sesgada a la derecha" if sk>0.5 else "sesgada a la izquierda" if sk<-0.5 else "aproximadamente simétrica"}')
print(f'Curtosis:           {ku:.3f}  → {"leptocúrtica (picos altos)" if ku>1 else "platocúrtica (picos bajos)" if ku<-1 else "mesocúrtica (normal)"}')
print()

# Detección de outliers con IQR
q1 = df['min_conexion_dia'].quantile(0.25)
q3 = df['min_conexion_dia'].quantile(0.75)
iqr = q3 - q1
outliers = df[(df['min_conexion_dia'] < q1-1.5*iqr) | (df['min_conexion_dia'] > q3+1.5*iqr)]
print(f'Outliers detectados (IQR): {len(outliers)}')
print(outliers[['usuario_id','min_conexion_dia']])

In [ ]:
# ✏️ Forma de la distribución — slide 23, pregunta 1:
r_forma = ""

# ✏️ Valores extremos — slide 23, pregunta 2:
r_extremos = ""

# ✏️ Decisiones analíticas — slide 23, pregunta 3:
r_decisiones = ""

print(f'Forma distribución: {r_forma}')
print(f'Valores extremos:   {r_extremos}')
print(f'Decisiones:         {r_decisiones}')

---
## PARTE 4 — Actividad autónoma: Análisis profundo de conexiones

### 4.1 Cargar datos

In [ ]:
# Código exacto de la presentación
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('conexiones_diarias.csv')
print(df.describe().round(2))

### 4.2 Histograma simple — código exacto presentación

In [ ]:
# Código exacto slide 28
sns.histplot(df['min_conexion_dia'], bins=8)
plt.title('Distribución de minutos de conexión por día')
plt.xlabel('Minutos')
plt.ylabel('Frecuencia')
plt.show()

### 4.3 Curva KDE — código exacto presentación

In [ ]:
# Código exacto slide 28
sns.kdeplot(df['min_conexion_dia'], fill=True)
plt.title('Curva de densidad de conexión diaria')
plt.xlabel('Minutos')
plt.ylabel('Densidad')
plt.show()

### 4.4 Visualización combinada — código exacto presentación

In [ ]:
# Código exacto slide 29
sns.histplot(df['min_conexion_dia'], bins=8, kde=True)
plt.title('Distribución con KDE combinada')
plt.xlabel('Minutos')
plt.ylabel('Frecuencia / Densidad')
plt.show()

### 4.5 Análisis adicional: distribución por día de la semana

In [ ]:
# KDE múltiple por día de la semana (variable adicional del dataset)
orden_dias = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
dias_presentes = [d for d in orden_dias if d in df['dia_semana'].unique()]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Conexión por día de la semana', fontweight='bold')

# KDE por día
for dia in dias_presentes:
    subset = df[df['dia_semana']==dia]['min_conexion_dia']
    if len(subset) > 3:
        sns.kdeplot(subset, fill=False, ax=axes[0], label=dia, linewidth=1.8)
axes[0].set_title('Curvas KDE por día de la semana')
axes[0].set_xlabel('Minutos')
axes[0].legend(fontsize=7)

# Boxplot por día
df_dias = df[df['dia_semana'].isin(dias_presentes)]
sns.boxplot(data=df_dias, x='dia_semana', y='min_conexion_dia',
            order=dias_presentes, palette='Blues_r', ax=axes[1])
axes[1].set_title('Boxplot por día de la semana')
axes[1].set_xlabel('')
axes[1].set_ylabel('Minutos')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 4.6 Panel completo de diagnóstico

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Panel de diagnóstico de distribución — Conexiones diarias',
             fontweight='bold', fontsize=12)

media   = df['min_conexion_dia'].mean()
mediana = df['min_conexion_dia'].median()

# 1. Histograma con media y mediana
sns.histplot(df['min_conexion_dia'], bins=15, kde=False,
             ax=axes[0,0], color='#2E75B6', alpha=0.8)
axes[0,0].axvline(media,   color='red',   linestyle='--', linewidth=2, label=f'Media {media:.1f}')
axes[0,0].axvline(mediana, color='green', linestyle='--', linewidth=2, label=f'Mediana {mediana:.1f}')
axes[0,0].set_title('Histograma con media y mediana')
axes[0,0].set_xlabel('Minutos')
axes[0,0].legend(fontsize=8)

# 2. KDE fill
sns.kdeplot(df['min_conexion_dia'], fill=True, bw_adjust=0.8,
            color='orange', ax=axes[0,1])
axes[0,1].axvline(media,   color='red',   linestyle='--', linewidth=2)
axes[0,1].axvline(mediana, color='green', linestyle='--', linewidth=2)
axes[0,1].set_title('Curva KDE (bw_adjust=0.8)')
axes[0,1].set_xlabel('Minutos')

# 3. Combinado
sns.histplot(df['min_conexion_dia'], bins=12, kde=True,
             color='purple', alpha=0.6, ax=axes[1,0])
axes[1,0].set_title('Combinado: histplot + kde')
axes[1,0].set_xlabel('Minutos')

# 4. Boxplot horizontal
sns.boxplot(x=df['min_conexion_dia'], ax=axes[1,1], color='#BDD7EE')
axes[1,1].set_title('Boxplot — outliers visibles')
axes[1,1].set_xlabel('Minutos')

plt.tight_layout()
plt.show()

### ✏️ Reflexiones — preguntas del slide 29:

In [ ]:
# ✏️ 1. Forma de la distribución — ¿simétrica, sesgada, multimodal?
c1 = ""

# ✏️ 2. ¿Qué representa la curva KDE en comparación con el histograma?
c2 = ""

# ✏️ 3. ¿Hay valores extremos o agrupamientos evidentes?
c3 = ""

# ✏️ 4. ¿Cómo orientarías a la coordinación académica basándote en estos datos?
c4 = ""

# ✏️ 5. ¿Qué gráfico fue más útil para tu análisis y por qué?
c5 = ""

print('--- REFLEXIONES ---')
for i, c in enumerate([c1, c2, c3, c4, c5], 1):
    print(f'{i}. {c}')

---
## 📋 Resumen de funciones y parámetros

### `histplot()` — parámetros clave

| Parámetro | Valores | Descripción |
|-----------|---------|-------------|
| `bins` | int | Número de barras (prueba 10-20 primero) |
| `stat` | `'count'`, `'probability'`, `'density'` | Qué muestran las barras |
| `kde` | `True`/`False` | Superpone curva de densidad |
| `color` | string | Color de las barras |
| `alpha` | 0.0–1.0 | Transparencia |
| `hue` | columna | Colorear por categoría |

### `kdeplot()` — parámetros clave

| Parámetro | Valores | Descripción |
|-----------|---------|-------------|
| `fill` | `True`/`False` | Rellenar bajo la curva |
| `bw_adjust` | float | Suavizado (0.5=fino, 2.0=suave) |
| `color` | string | Color de la curva |
| `hue` | columna | Múltiples curvas por categoría |
| `cumulative` | `True`/`False` | Distribución acumulada |

**Cuándo usar cada uno:**

| Situación | Usar |
|-----------|------|
| Ver conteos en rangos | `histplot(stat='count')` |
| Comparar proporciones | `histplot(stat='probability')` |
| Ver forma continua de distribución | `kdeplot(fill=True)` |
| Comparar grupos en una misma escala | `kdeplot(hue='grupo')` |
| Vista completa de forma y conteos | `histplot(kde=True)` |

> 💡 **`distplot` está DEPRECADO** desde Seaborn 0.11. Siempre usa `histplot()` + `kdeplot()` por separado o `histplot(kde=True)` para la combinación.

> 💡 **Interpretación de sesgo:** si `media > mediana` → sesgo positivo (cola a la derecha). Si `media < mediana` → sesgo negativo (cola a la izquierda).